# 05 How well does Valentine recover documented relationships?

The core measurement: for each documented table pair, does each of Valentine's built-in
matchers put the documented column in the top-k for its source column?

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / "src"))
import pandas as pd
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 160)

In [2]:
results = pd.read_csv("../evaluation/results_attribution.csv")
print(f"{len(results)} runs, {int((results.status != 'ok').sum())} errors")
results.groupby(["matcher", "status"]).size().unstack(fill_value=0)

300 runs, 12 errors


status,error,ok
matcher,,
coma,0,60
cupid,0,60
distribution_based,12,48
jaccard,0,60
similarity_flooding,0,60


## Recovery@1 by level, by matcher (attribution mode)

Each level applies only its own operators to the original data, so levels do not stack and
a later level can recover (COMA: 0.542 at L1, back to 0.958 at L2). L0 is the uncorrupted
baseline. Cumulative results are in notebook 06.

The `recovery_at_*_degenerate` flags mark runs where the target table has so few columns
that top-k contains everything. Those runs are excluded from the means — a column of 1.0s
the matcher did not earn would flatter every matcher equally and hide the differences the
table exists to show.

In [3]:
from swsn.run_benchmark import summary

summary(results).to_string()

'matcher   coma  cupid  distribution_based  jaccard  similarity_flooding\nlevel                                                                  \nL0       0.958  0.847               0.648    0.875                0.958\nL1       0.542  0.292               0.537    0.708                0.833\nL2       0.958  0.458               0.273    0.569                0.958\nL3       0.958  0.542               0.200    0.681                0.958\nL4       0.958  0.847               0.648    0.875                0.958'

## Per-document table pair, at L0

In [4]:
l0 = results[(results.level == "L0") & (results.status == "ok")]
l0.pivot_table(index=["dataset", "pair_id"], columns="matcher",
               values="recovery_at_1", aggfunc="first").round(2)

matcher                                    coma  cupid  distribution_based  jaccard  similarity_flooding
dataset   pair_id                                                                                       
instacart instacart_orders__prior           1.0   1.00                0.00      1.0                  1.0
          instacart_prior__train            1.0   1.00                0.00      0.0                  1.0
          instacart_products__aisles        1.0   1.00                1.00      1.0                  1.0
          instacart_products__departments   1.0   1.00                1.00      1.0                  1.0
olist     olist_customers__geolocation      1.0   0.33                0.33      1.0                  1.0
          olist_customers__sellers          1.0   0.33                1.00      1.0                  1.0
          olist_items__products             1.0   1.00                1.00      1.0                  1.0
          olist_items__sellers              1.0   1.00                1.00      1.0                  1.0
          olist_orders__customers           1.0   1.00                 NaN      1.0                  1.0
          olist_orders__items               1.0   1.00                 NaN      1.0                  1.0
          olist_orders__payments            1.0   1.00                 NaN      1.0                  1.0
          olist_products__translation       0.5   0.50                0.50      0.5                  0.5

## DistributionBased does not run reliably here

12 of 300 runs raise, all from this one matcher, in two distinct ways:

- `PulpSolverError` from `COIN_CMD` — reproducible on the same input. The bundled CBC
  binary runs fine standalone, so it is the invocation.
- `ValueError: malformed node` from `literal_eval(v.name)` — the solver's variable names are
  built as `"(order_id,payment_type)"` and parsed back as a Python tuple, which only works
  when the vertex names are numeric.

valentine 1.1.0 declares `PuLP[cbc]>=4.0,<5.0`, so this is a defect against its own declared
dependency, not a pinning mistake on our side. It stays in the grid with its failure count
reported and is excluded from cross-matcher comparison.